# Outbox Pattern

Реалізація патерну **Transactional Outbox** для надійної публікації доменних подій.

**Проблема:** при збої між записом у БД та публікацією в чергу (RabbitMQ / Azure Service Bus)
повідомлення губиться — порушується гарантія доставки.

**Рішення Outbox:** зберігати повідомлення в таблицю `outbox_messages` **в тій самій транзакції**,
що й бізнес-дані. Окремий фоновий процес (OutboxProcessor) читає таблицю і публікує повідомлення.

```
Order.create()
      ↓
BEGIN TRANSACTION
  INSERT orders
  INSERT outbox_messages   ← атомарно разом
COMMIT
      ↓
OutboxProcessor (background)
  SELECT unprocessed outbox_messages
  publish → broker
  UPDATE processed_at
```

## 1. Доменна модель

In [ ]:
from __future__ import annotations

import json
import logging
import sqlite3
from dataclasses import dataclass, field
from datetime import datetime, timezone
from decimal import Decimal
from uuid import UUID, uuid4

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
log = logging.getLogger(__name__)


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


print("Imports OK")

In [ ]:
# --- Domain event ---

@dataclass(frozen=True)
class OrderCreated:
    """Domain event emitted when an order is successfully created."""

    order_id: UUID
    customer_id: str
    total_amount: Decimal
    created_at: str


# --- Aggregate ---

@dataclass
class Order:
    """Order aggregate — collects domain events via create()."""

    order_id: UUID
    customer_id: str
    total_amount: Decimal
    created_at: str
    _pending_events: list[OrderCreated] = field(default_factory=list, repr=False)

    @classmethod
    def create(cls, customer_id: str, total_amount: Decimal) -> Order:
        """Factory: validates, creates the aggregate, emits OrderCreated."""
        if total_amount <= 0:
            raise ValueError("Сума замовлення має бути > 0")
        order = cls(
            order_id=uuid4(),
            customer_id=customer_id,
            total_amount=total_amount,
            created_at=now_iso(),
        )
        order._pending_events.append(
            OrderCreated(
                order_id=order.order_id,
                customer_id=order.customer_id,
                total_amount=order.total_amount,
                created_at=order.created_at,
            )
        )
        return order

    @property
    def pending_events(self) -> list[OrderCreated]:
        return list(self._pending_events)

    def clear_events(self) -> None:
        self._pending_events.clear()


# Quick smoke test
order = Order.create("cust-1", Decimal("250.00"))
assert len(order.pending_events) == 1
assert order.pending_events[0].total_amount == Decimal("250.00")
print(f"Order created: {order.order_id}")
print(f"Pending events: {[type(e).__name__ for e in order.pending_events]}")

## 2. Outbox Pattern

### 2.1 Модель OutboxMessage та схема БД

In [ ]:
@dataclass
class OutboxMessage:
    """Outbox table row — one message per domain event."""

    id: UUID
    event_type: str          # e.g. "OrderCreated"
    payload: str             # JSON-serialized event
    created_at: str          # ISO timestamp
    processed_at: str | None = None
    retry_count: int = 0


def create_db() -> sqlite3.Connection:
    """Create in-memory SQLite DB with orders, outbox_messages, dead_letter tables."""
    conn = sqlite3.connect(":memory:")
    conn.executescript("""
        CREATE TABLE orders (
            order_id    TEXT PRIMARY KEY,
            customer_id TEXT NOT NULL,
            total_amount TEXT NOT NULL,
            created_at  TEXT NOT NULL
        );

        CREATE TABLE outbox_messages (
            id           TEXT PRIMARY KEY,
            event_type   TEXT NOT NULL,
            payload      TEXT NOT NULL,
            created_at   TEXT NOT NULL,
            processed_at TEXT,
            retry_count  INTEGER NOT NULL DEFAULT 0
        );

        CREATE TABLE dead_letter (
            id          TEXT PRIMARY KEY,
            event_type  TEXT NOT NULL,
            payload     TEXT NOT NULL,
            created_at  TEXT NOT NULL,
            failed_at   TEXT NOT NULL,
            retry_count INTEGER NOT NULL
        );
    """)
    return conn


print("OutboxMessage and DB schema defined")

### 2.2 UnitOfWork — транзакційний запис (SaveChanges)

Python-еквівалент `DbContext.SaveChangesAsync` з C#:
замовлення та outbox-повідомлення зберігаються **в одній транзакції** — або обидва, або жоден.

In [ ]:
class UnitOfWork:
    """Coordinates transactional writes of domain objects + outbox messages."""

    def __init__(self, conn: sqlite3.Connection) -> None:
        self._conn = conn

    def save(self, order: Order) -> None:
        """Persist order and its domain events to outbox in a single transaction."""
        with self._conn:
            # Insert business data
            self._conn.execute(
                "INSERT INTO orders (order_id, customer_id, total_amount, created_at)"
                " VALUES (?, ?, ?, ?)",
                (str(order.order_id), order.customer_id,
                 str(order.total_amount), order.created_at),
            )

            # Insert one outbox row per domain event
            for event in order.pending_events:
                payload = json.dumps({
                    "order_id": str(event.order_id),
                    "customer_id": event.customer_id,
                    "total_amount": str(event.total_amount),
                    "created_at": event.created_at,
                })
                self._conn.execute(
                    "INSERT INTO outbox_messages (id, event_type, payload, created_at)"
                    " VALUES (?, ?, ?, ?)",
                    (str(uuid4()), type(event).__name__, payload, now_iso()),
                )

            order.clear_events()
            log.info("Saved order %s with outbox message", order.order_id)


print("UnitOfWork defined")

### 2.3 OutboxProcessor — фонова публікація (BackgroundService)

Python-еквівалент `BackgroundService` з C#: читає необроблені повідомлення з outbox,
публікує їх до брокера повідомлень (Message Broker) або симулює публікацію, оновлює `processed_at`.

**Логіка повторних спроб (бонус):** якщо публікація падає — збільшує `retry_count`.
При `retry_count >= 3` повідомлення переміщується до таблиці `dead_letter`.

In [ ]:
MAX_RETRIES = 3


class OutboxProcessor:
    """Polls outbox_messages and publishes them; handles retries and dead letter."""

    def __init__(self, conn: sqlite3.Connection, publisher) -> None:
        """
        publisher: callable(event_type: str, payload: str) -> None
            Raises on failure (simulates broker unavailability).
        """
        self._conn = conn
        self._publisher = publisher

    def process_batch(self) -> int:
        """Process all pending outbox messages. Returns count of messages handled."""
        rows = self._conn.execute(
            "SELECT id, event_type, payload, created_at, retry_count"
            " FROM outbox_messages"
            " WHERE processed_at IS NULL AND retry_count < ?",
            (MAX_RETRIES,),
        ).fetchall()

        handled = 0
        for msg_id, event_type, payload, created_at, retry_count in rows:
            try:
                self._publisher(event_type, payload)
                # Mark as processed
                with self._conn:
                    self._conn.execute(
                        "UPDATE outbox_messages SET processed_at = ? WHERE id = ?",
                        (now_iso(), msg_id),
                    )
                log.info("Published %s [%s]", event_type, msg_id)
                handled += 1
            except Exception as exc:
                new_retry = retry_count + 1
                if new_retry >= MAX_RETRIES:
                    # Move to dead letter
                    with self._conn:
                        self._conn.execute(
                            "INSERT INTO dead_letter"
                            " (id, event_type, payload, created_at, failed_at, retry_count)"
                            " VALUES (?, ?, ?, ?, ?, ?)",
                            (msg_id, event_type, payload, created_at, now_iso(), new_retry),
                        )
                        self._conn.execute(
                            "DELETE FROM outbox_messages WHERE id = ?",
                            (msg_id,),
                        )
                    log.warning("Dead letter: %s [%s] after %d retries", event_type, msg_id, new_retry)
                else:
                    with self._conn:
                        self._conn.execute(
                            "UPDATE outbox_messages SET retry_count = ? WHERE id = ?",
                            (new_retry, msg_id),
                        )
                    log.warning("Retry %d/%d for %s [%s]: %s", new_retry, MAX_RETRIES, event_type, msg_id, exc)

        return handled


print("OutboxProcessor defined")

## 3. Демонстрація

### 3.1 Успішний сценарій (Happy path) — створення замовлення та публікація

In [ ]:
published: list[tuple[str, str]] = []

def ok_publisher(event_type: str, payload: str) -> None:
    """Simulates a successful broker publish."""
    published.append((event_type, payload))


db = create_db()
uow = UnitOfWork(db)
processor = OutboxProcessor(db, ok_publisher)

# 1. Create order (domain event pending)
order = Order.create("cust-42", Decimal("499.00"))

# 2. Save — atomically writes order + outbox message
uow.save(order)

# Verify: order in DB, outbox has 1 unprocessed message
order_row = db.execute("SELECT * FROM orders WHERE order_id = ?", (str(order.order_id),)).fetchone()
outbox_row = db.execute("SELECT * FROM outbox_messages WHERE processed_at IS NULL").fetchone()
assert order_row is not None
assert outbox_row is not None
print(f"Order saved: {order_row[0]}")
print(f"Outbox pending: event_type={outbox_row[1]}, retry_count={outbox_row[5]}")

# 3. Processor publishes
count = processor.process_batch()
assert count == 1
assert len(published) == 1
assert published[0][0] == "OrderCreated"

# Verify: outbox message is now marked processed
processed_row = db.execute("SELECT processed_at FROM outbox_messages").fetchone()
assert processed_row[0] is not None
print(f"Published: {published[0][0]}")
print(f"Outbox processed_at: {processed_row[0]}")
print("Happy path: OK")

### 3.2 Атомарність — збій при збереженні

Якщо транзакція відкочується — ні замовлення, ні outbox-повідомлення не з'являються в БД.

In [ ]:
class FailingUnitOfWork(UnitOfWork):
    """UnitOfWork that always raises after inserting the order (simulates mid-save crash)."""

    def save(self, order: Order) -> None:
        with self._conn:
            self._conn.execute(
                "INSERT INTO orders (order_id, customer_id, total_amount, created_at)"
                " VALUES (?, ?, ?, ?)",
                (str(order.order_id), order.customer_id,
                 str(order.total_amount), order.created_at),
            )
            raise RuntimeError("Симуляція збою БД після INSERT order")


db2 = create_db()
failing_uow = FailingUnitOfWork(db2)

bad_order = Order.create("cust-bad", Decimal("100.00"))
try:
    failing_uow.save(bad_order)
except RuntimeError as e:
    print(f"Caught: {e}")

# Verify: neither order nor outbox row exists (full rollback)
orders_count = db2.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
outbox_count = db2.execute("SELECT COUNT(*) FROM outbox_messages").fetchone()[0]
assert orders_count == 0, f"Expected 0 orders, got {orders_count}"
assert outbox_count == 0, f"Expected 0 outbox rows, got {outbox_count}"
print(f"Orders in DB: {orders_count}  (expected 0)")
print(f"Outbox rows:  {outbox_count}  (expected 0)")
print("Atomicity: OK — rollback is total")

## 4. Бонус: Повторні спроби (Retry) та Dead Letter

Якщо публікація до брокера повідомлень завжди падає — OutboxProcessor збільшує `retry_count`.
Після `MAX_RETRIES` спроб (3) повідомлення переміщується до таблиці `dead_letter`.

In [ ]:
db3 = create_db()
uow3 = UnitOfWork(db3)

def always_fails(event_type: str, payload: str) -> None:
    """Simulates a broker that is permanently unavailable."""
    raise ConnectionError("Брокер недоступний")

failing_processor = OutboxProcessor(db3, always_fails)

# Create and save an order
order3 = Order.create("cust-retry", Decimal("75.00"))
uow3.save(order3)

print(f"MAX_RETRIES = {MAX_RETRIES}")

# Attempt 1 — retry_count becomes 1
failing_processor.process_batch()
rc = db3.execute("SELECT retry_count FROM outbox_messages").fetchone()
print(f"After attempt 1: retry_count = {rc[0]}")
assert rc[0] == 1

# Attempt 2 — retry_count becomes 2
failing_processor.process_batch()
rc = db3.execute("SELECT retry_count FROM outbox_messages").fetchone()
print(f"After attempt 2: retry_count = {rc[0]}")
assert rc[0] == 2

# Attempt 3 — retry_count reaches MAX_RETRIES → moves to dead_letter
failing_processor.process_batch()

outbox_remaining = db3.execute("SELECT COUNT(*) FROM outbox_messages").fetchone()[0]
dead_letter_count = db3.execute("SELECT COUNT(*) FROM dead_letter").fetchone()[0]
dl_row = db3.execute("SELECT event_type, retry_count FROM dead_letter").fetchone()

assert outbox_remaining == 0, f"Expected 0 outbox rows, got {outbox_remaining}"
assert dead_letter_count == 1, f"Expected 1 dead letter row, got {dead_letter_count}"
assert dl_row[1] == MAX_RETRIES

print(f"Outbox remaining:   {outbox_remaining}  (expected 0)")
print(f"Dead letter rows:   {dead_letter_count}  (expected 1)")
print(f"Dead letter event:  {dl_row[0]}, retry_count={dl_row[1]}")
print("Retry + Dead Letter: OK")

## Архітектура

### Потік даних

```
Order.create()
  └─ OrderCreated (domain event, pending)
              ↓
        UnitOfWork.save(order)
              ↓
        BEGIN TRANSACTION
          INSERT orders
          INSERT outbox_messages  ← атомарно
        COMMIT
              ↓
        OutboxProcessor.process_batch()
          SELECT outbox WHERE processed_at IS NULL AND retry_count < 3
          publish(event_type, payload)
          ├─ success → UPDATE processed_at
          └─ failure → retry_count++
                         └─ >= MAX_RETRIES → dead_letter
```

### Гарантії

| Сценарій | Результат |
|---|---|
| Збій до COMMIT | Ні замовлення, ні outbox-повідомлення |
| Збій після COMMIT, до публікації | Повідомлення залишається в outbox → процесор опублікує при наступному запуску |
| Збій публікації (broker down) | retry_count++ до MAX_RETRIES, потім dead_letter |
| Успішна публікація | processed_at виставлено, повідомлення більше не обробляється |